# NB04: Per-Function Cultivation Coverage Analysis

Compare KO profiles between cultured genomes and MAGs to quantify per-function
cultivation bias. Tests H1 (depletion) and H2 (enrichment) using Fisher's exact test.

**Requires**:
- `data/cultured_ko_profiles.tsv` — from NB03 (JupyterHub Spark query of ENIGMA Genome Depot)
- `data/mag_ko_profiles.tsv` — extracted from `kbase_ke_pangenome.bakta_db_xrefs` via Spark

### MAG KO Profile Generation

The MAG cohort was pivoted from CORAL ORFRC-specific MAGs (inaccessible assembly FASTAs)
to 2,279 QC-passing subsurface/groundwater MAGs from `kbase_ke_pangenome`. KO profiles
were extracted via the following Spark SQL pipeline:

1. Identify subsurface MAGs: `kbase_ke_pangenome.gtdb_metadata` (ncbi_genome_category = "derived from metagenome") joined to `ncbi_env` (isolation sources matching subsurface/groundwater/aquifer), filtered for completeness >= 50%, contamination <= 10%
2. Extract KEGG KOs: `gene` -> `gene_genecluster_junction` -> `bakta_db_xrefs` (db = "KEGG"), grouped by genome_id and ko_id

See RESEARCH_PLAN.md v2 for full rationale behind the cohort pivot.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

## 1. Load Both Cohorts

In [ ]:
cult_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
mag_ko = pd.read_csv(DATA_DIR / 'mag_ko_profiles.tsv', sep='\t')

n_cult = cult_ko['genome_id'].nunique()
n_mag = mag_ko['genome_id'].nunique()
print(f'Cultured: {n_cult} genomes, {cult_ko["ko_id"].nunique()} unique KOs')
print(f'MAGs:     {n_mag} genomes, {mag_ko["ko_id"].nunique()} unique KOs')

all_kos = set(cult_ko['ko_id'].unique()) | set(mag_ko['ko_id'].unique())
shared_kos = set(cult_ko['ko_id'].unique()) & set(mag_ko['ko_id'].unique())
cult_only = set(cult_ko['ko_id'].unique()) - set(mag_ko['ko_id'].unique())
mag_only = set(mag_ko['ko_id'].unique()) - set(cult_ko['ko_id'].unique())

print(f'\nKO overlap: {len(shared_kos)} shared, {len(cult_only)} cultured-only, {len(mag_only)} MAG-only')

## 2. Per-KO Prevalence (fraction of genomes carrying each KO)

In [ ]:
# Presence/absence per genome (ignore copy number for prevalence)
cult_prev = cult_ko.groupby('ko_id')['genome_id'].nunique().rename('n_cult')
mag_prev = mag_ko.groupby('ko_id')['genome_id'].nunique().rename('n_mag')

prev = pd.DataFrame(index=sorted(all_kos))
prev['n_cult'] = cult_prev.reindex(prev.index).fillna(0).astype(int)
prev['n_mag'] = mag_prev.reindex(prev.index).fillna(0).astype(int)
prev['frac_cult'] = prev['n_cult'] / n_cult
prev['frac_mag'] = prev['n_mag'] / n_mag
prev['log2_ratio'] = np.log2((prev['frac_cult'] + 0.001) / (prev['frac_mag'] + 0.001))

print(f'{len(prev)} KOs in combined set')
prev.head()

## 3. Fisher's Exact Test (per-KO enrichment/depletion)

In [ ]:
pvals = []
odds_ratios = []

for ko_id in prev.index:
    a = prev.loc[ko_id, 'n_cult']      # cultured with KO
    b = n_cult - a                       # cultured without KO
    c = prev.loc[ko_id, 'n_mag']        # MAG with KO
    d = n_mag - c                        # MAG without KO
    
    table = [[a, b], [c, d]]
    odds, p = stats.fisher_exact(table)
    pvals.append(p)
    odds_ratios.append(odds)

prev['pval'] = pvals
prev['odds_ratio'] = odds_ratios

# BH correction
reject, qvals, _, _ = multipletests(prev['pval'], method='fdr_bh')
prev['qval'] = qvals
prev['significant'] = reject

sig = prev[prev['significant']]
enriched = sig[sig['log2_ratio'] > 0]
depleted = sig[sig['log2_ratio'] < 0]

print(f'Significant KOs (FDR < 0.05): {len(sig)}/{len(prev)}')
print(f'  Enriched in cultured: {len(enriched)}')
print(f'  Depleted in cultured: {len(depleted)}')

## 4. Marker Dictionary Results

In [ ]:
markers = {
    'Wood-Ljungdahl': ['K00198', 'K00194', 'K00197', 'K15022', 'K15023'],
    'NiFe-hydrogenase': ['K06281', 'K06282'],
    'Sulfate reduction': ['K11180', 'K11181', 'K00394', 'K00395', 'K00958'],
    'Denitrification': ['K00370', 'K00371', 'K02567', 'K00368', 'K15864', 'K00376'],
    'Methanogenesis': ['K00399', 'K00401', 'K00402'],
    'N-fixation': ['K02588', 'K02586', 'K02591'],
    'Metal resistance': ['K07787', 'K07665', 'K19592', 'K00537'],
    'Motility/chemotaxis': ['K02406', 'K03406', 'K03407'],
    'Aerobic respiration': ['K02274', 'K02275', 'K02276'],
    'Conjugation/MGE': ['K03204', 'K03197'],
}

marker_results = []
for category, kos in markers.items():
    for ko in kos:
        if ko in prev.index:
            row = prev.loc[ko]
            marker_results.append({
                'category': category, 'ko_id': ko,
                'frac_cult': row['frac_cult'], 'frac_mag': row['frac_mag'],
                'log2_ratio': row['log2_ratio'], 'qval': row['qval'],
                'significant': row['significant']
            })
        else:
            marker_results.append({
                'category': category, 'ko_id': ko,
                'frac_cult': 0, 'frac_mag': 0,
                'log2_ratio': 0, 'qval': 1, 'significant': False
            })

mdf = pd.DataFrame(marker_results)
print(f'Marker dictionary results:')
print(f'{"Category":<25} {"Cult%":>7} {"MAG%":>7} {"log2R":>7} {"Sig?":>5}')
print('-' * 55)
for cat in markers:
    sub = mdf[mdf['category'] == cat]
    mean_cult = sub['frac_cult'].mean() * 100
    mean_mag = sub['frac_mag'].mean() * 100
    mean_lr = sub['log2_ratio'].mean()
    n_sig = sub['significant'].sum()
    print(f'{cat:<25} {mean_cult:>6.1f}% {mean_mag:>6.1f}% {mean_lr:>+7.2f} {n_sig}/{len(sub)}')

mdf.to_csv(DATA_DIR / 'marker_cultivation_coverage.tsv', sep='\t', index=False)

## 5. Volcano Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# All KOs
ns = prev[~prev['significant']]
ax.scatter(ns['log2_ratio'], -np.log10(ns['qval']), 
           c='lightgray', s=3, alpha=0.5, label='NS')

# Significant enriched (cultured)
ax.scatter(enriched['log2_ratio'], -np.log10(enriched['qval']),
           c='#d62728', s=5, alpha=0.7, label=f'Enriched in cultured (n={len(enriched)})')

# Significant depleted (cultured)
ax.scatter(depleted['log2_ratio'], -np.log10(depleted['qval']),
           c='#1f77b4', s=5, alpha=0.7, label=f'Depleted in cultured (n={len(depleted)})')

# Highlight marker KOs
marker_kos_flat = {ko: cat for cat, kos in markers.items() for ko in kos}
for ko, cat in marker_kos_flat.items():
    if ko in prev.index:
        row = prev.loc[ko]
        ax.scatter(row['log2_ratio'], -np.log10(row['qval']),
                  c='black', s=30, marker='D', zorder=5)
        if row['significant']:
            ax.annotate(ko, (row['log2_ratio'], -np.log10(row['qval'])),
                       fontsize=6, ha='left', va='bottom')

ax.axhline(-np.log10(0.05), ls='--', c='gray', lw=0.5)
ax.axvline(0, ls='-', c='gray', lw=0.5)
ax.set_xlabel('log2(cultured prevalence / MAG prevalence)')
ax.set_ylabel('-log10(q-value)')
ax.set_title('Cultivation Bias: Cultured vs MAG KO Prevalence')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'volcano_cultivation_bias.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Marker Category Heatmap

In [ ]:
# Category-level summary heatmap
cat_summary = []
for cat in markers:
    sub = mdf[mdf['category'] == cat]
    cat_summary.append({
        'category': cat,
        'mean_frac_cult': sub['frac_cult'].mean(),
        'mean_frac_mag': sub['frac_mag'].mean(),
        'mean_log2_ratio': sub['log2_ratio'].mean(),
        'n_sig': sub['significant'].sum(),
        'n_kos': len(sub)
    })

cat_df = pd.DataFrame(cat_summary).set_index('category')

fig, ax = plt.subplots(figsize=(8, 5))
hm_data = cat_df[['mean_frac_cult', 'mean_frac_mag']].rename(
    columns={'mean_frac_cult': 'Cultured', 'mean_frac_mag': 'MAG'})

sns.heatmap(hm_data, annot=True, fmt='.2f', cmap='YlOrRd',
            ax=ax, vmin=0, vmax=1)
ax.set_title('Mean KO Prevalence by Marker Category')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(FIG_DIR / 'marker_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Export Full Results

In [ ]:
prev.to_csv(DATA_DIR / 'ko_cultivation_coverage_full.tsv', sep='\t')
print(f'Saved full coverage table: {len(prev)} KOs')
print(f'\nTop 20 most cultured-depleted KOs (by log2 ratio, q<0.05):')
top_depleted = prev[prev['significant']].nsmallest(20, 'log2_ratio')
for ko, row in top_depleted.iterrows():
    print(f'  {ko}: cult={row["frac_cult"]:.3f} mag={row["frac_mag"]:.3f} log2R={row["log2_ratio"]:.2f}')

print(f'\nTop 20 most cultured-enriched KOs (by log2 ratio, q<0.05):')
top_enriched = prev[prev['significant']].nlargest(20, 'log2_ratio')
for ko, row in top_enriched.iterrows():
    print(f'  {ko}: cult={row["frac_cult"]:.3f} mag={row["frac_mag"]:.3f} log2R={row["log2_ratio"]:.2f}')